# LLM Serving and Inference Optimization
## Live Demo — Instructor-Run
**ML System Design and LLMOps | Week 3, Session 1**

Today we shift to large language models. The engineering challenges are qualitatively different from classical ML: models are orders of magnitude larger, inference is compute-bound not IO-bound, and the cost structure means every optimization decision has direct dollar implications.

We'll work through four concrete things:
1. Memory calculator — how much GPU RAM does any given model actually need?
2. Cost estimator — API cost vs self-hosted GPU cost, and where the break-even is
3. Quantization demo — what int8/fp16 actually does to model size and speed
4. KV cache illustration — why caching attention states is the single biggest serving optimization

## Part 1: LLM Memory Calculator
*"Can we fit this model on an A100?" — let's compute it properly*

We're going to build a function that answers this question for any model size. The formula is simple: `params × bytes_per_param`. But it's easy to forget activations, optimizer states, and the KV cache — so we'll account for all of them.

In [ ]:
def llm_memory_gb(params_billions: float, dtype: str = "fp16") -> float:
    """
    Compute memory requirement for model weights only.
    
    dtype options:
      fp32  → 4 bytes per parameter
      fp16  → 2 bytes per parameter  (default for inference)
      bf16  → 2 bytes per parameter  (same as fp16 for memory)
      int8  → 1 byte per parameter   (quantized)
      int4  → 0.5 bytes per parameter (aggressively quantized)
    """
    bytes_per_param = {"fp32": 4, "fp16": 2, "bf16": 2, "int8": 1, "int4": 0.5}
    if dtype not in bytes_per_param:
        raise ValueError(f"Unknown dtype: {dtype}. Choose from {list(bytes_per_param.keys())}")
    
    params = params_billions * 1e9
    total_bytes = params * bytes_per_param[dtype]
    return total_bytes / 1e9  # convert to GB


# Models and their parameter counts (in billions)
models = {
    "GPT-3":     175.0,
    "Llama-7B":    7.0,
    "Llama-70B":  70.0,
    "Llama-3-8B":  8.0,
    "Mistral-7B":  7.0,
}

print("=== LLM Memory Calculator ===")
print()
print("GPU reference:")
print("  A100-40GB: 40 GB VRAM")
print("  A100-80GB: 80 GB VRAM")
print("  H100-80GB: 80 GB VRAM")
print()

# Table header
print(f"┌{'─'*17}┬{'─'*12}┬{'─'*14}┬{'─'*14}┬{'─'*14}┐")
print(f"│ {'Model':<15} │ {'Params':>10} │ {'fp32 (GB)':>12} │ {'fp16 (GB)':>12} │ {'int8 (GB)':<12} │")
print(f"├{'─'*17}┼{'─'*12}┼{'─'*14}┼{'─'*14}┼{'─'*14}┤")

for name, params_b in models.items():
    fp32_gb = llm_memory_gb(params_b, "fp32")
    fp16_gb = llm_memory_gb(params_b, "fp16")
    int8_gb = llm_memory_gb(params_b, "int8")
    print(f"│ {name:<15} │ {params_b:>8.1f} B  │ {fp32_gb:>10.1f} GB │ {fp16_gb:>10.1f} GB │ {int8_gb:>8.1f} GB   │")

print(f"└{'─'*17}┴{'─'*12}┴{'─'*14}┴{'─'*14}┴{'─'*14}┘")

print()
print("int4 (aggressive quantization):")
for name, params_b in models.items():
    int4_gb = llm_memory_gb(params_b, "int4")
    note = ""
    if int4_gb < 24:
        note = " ← fits on a consumer RTX 4090 (24GB)!"
    elif int4_gb < 40:
        note = " ← fits on a single A100-40GB!"
    print(f"  {name:<12}: {int4_gb:>6.1f} GB{note}")

print()
print("Note: these are model weights only.")
print("Add ~20% for activations and KV cache during inference.")
print("For training: add 8x (fp32 optimizer states + gradients + activations).")

=== LLM Memory Calculator ===

GPU reference:
  A100-40GB: 40 GB VRAM
  A100-80GB: 80 GB VRAM
  H100-80GB: 80 GB VRAM

┌─────────────────┬────────────┬──────────────┬──────────────┬──────────────┐
│ Model           │   Params   │  fp32 (GB)   │  fp16 (GB)   │ int8 (GB)    │
├─────────────────┼────────────┼──────────────┼──────────────┼──────────────┤
│ GPT-3           │   175.0 B  │   700.0 GB   │   350.0 GB   │   175.0 GB   │
│ Llama-7B        │     7.0 B  │    28.0 GB   │    14.0 GB   │     7.0 GB   │
│ Llama-70B       │    70.0 B  │   280.0 GB   │   140.0 GB   │    70.0 GB   │
│ Llama-3-8B      │     8.0 B  │    32.0 GB   │    16.0 GB   │     8.0 GB   │
│ Mistral-7B      │     7.0 B  │    28.0 GB   │    14.0 GB   │     7.0 GB   │
└─────────────────┴────────────┴──────────────┴──────────────┴──────────────┘

int4 (aggressive quantization):
  GPT-3:      87.5 GB
  Llama-7B:    3.5 GB  ← fits on a consumer RTX 4090 (24GB)!
  Llama-70B:  35.0 GB  ← fits on a single A100-40GB!

Note: the

---
## Part 2: Cost Estimator — API vs Self-Hosted
*The build-vs-buy decision in numbers*

Every team deploying an LLM faces this question: do we call the OpenAI API, or do we run our own GPU cluster? We're going to compute the break-even point.

In [ ]:
def compute_llm_costs(
    requests_per_day: int,
    avg_input_tokens: int,
    avg_output_tokens: int,
    api_input_cost_per_1m: float,   # $ per 1M input tokens
    api_output_cost_per_1m: float,  # $ per 1M output tokens
    gpu_hourly_rate: float,          # $ per hour for GPU instance
    gpu_active_hours_per_day: float = 24,
):
    """
    Compare API cost vs self-hosted GPU cost for an LLM workload.
    Returns a dict of all cost metrics.
    """
    # Token volumes
    total_input_per_day = requests_per_day * avg_input_tokens
    total_output_per_day = requests_per_day * avg_output_tokens
    total_tokens_per_day = total_input_per_day + total_output_per_day
    total_tokens_per_month = total_tokens_per_day * 30

    # API costs
    daily_api_cost = (
        (total_input_per_day / 1e6) * api_input_cost_per_1m +
        (total_output_per_day / 1e6) * api_output_cost_per_1m
    )
    monthly_api_cost = daily_api_cost * 30
    annual_api_cost = daily_api_cost * 365

    # GPU costs (always-on)
    monthly_gpu_cost_always_on = gpu_hourly_rate * 24 * 30
    # GPU costs (optimized: only pay for active hours)
    monthly_gpu_cost_optimized = gpu_hourly_rate * gpu_active_hours_per_day * 30

    # Break-even: how many tokens/month makes self-hosted cheaper?
    # API cost = (input_tokens/1M * input_price) + (output_tokens/1M * output_price)
    # Assume 70/30 input/output split
    effective_cost_per_token = (
        0.7 * api_input_cost_per_1m + 0.3 * api_output_cost_per_1m
    ) / 1e6
    breakeven_tokens_per_month = monthly_gpu_cost_optimized / effective_cost_per_token
    breakeven_requests_per_day = breakeven_tokens_per_month / 30 / (avg_input_tokens + avg_output_tokens)

    return {
        "total_tokens_per_day": total_tokens_per_day,
        "total_tokens_per_month": total_tokens_per_month,
        "daily_api_cost": daily_api_cost,
        "monthly_api_cost": monthly_api_cost,
        "annual_api_cost": annual_api_cost,
        "monthly_gpu_always_on": monthly_gpu_cost_always_on,
        "annual_gpu_always_on": monthly_gpu_cost_always_on * 12,
        "monthly_gpu_optimized": monthly_gpu_cost_optimized,
        "annual_gpu_optimized": monthly_gpu_cost_optimized * 12,
        "breakeven_tokens_per_month": breakeven_tokens_per_month,
        "breakeven_requests_per_day": breakeven_requests_per_day,
    }


# ── Worked example ────────────────────────────────────────────────────────────
costs = compute_llm_costs(
    requests_per_day=10_000,
    avg_input_tokens=500,
    avg_output_tokens=200,
    api_input_cost_per_1m=2.50,    # GPT-4o input pricing (approx, mid-2025)
    api_output_cost_per_1m=10.00,  # GPT-4o output pricing
    gpu_hourly_rate=32.77,          # AWS p4d.24xlarge (8x A100-80GB)
    gpu_active_hours_per_day=12,    # optimized: scale down at night
)

# Theoretical throughput for sizing utilization
# A100-80GB can process ~40K tokens/sec for Llama-70B in fp16
theoretical_tokens_per_month = 40_000 * 3600 * 24 * 30
utilization = costs["total_tokens_per_month"] / theoretical_tokens_per_month

print("=== LLM Cost Estimator ===")
print()
print("--- Use Case Parameters ---")
print(f"  Requests per day:        {10_000:,}")
print(f"  Avg input tokens:        500")
print(f"  Avg output tokens:       200")
print(f"  Total tokens per day:    {costs['total_tokens_per_day']:,}")
print(f"  Total tokens per month:  {costs['total_tokens_per_month']:,}")
print()
print("--- API Cost (GPT-4o pricing, July 2025) ---")
print(f"  Input:  $2.50 per 1M tokens")
print(f"  Output: $10.00 per 1M tokens")
print(f"  Daily cost:    ${costs['daily_api_cost']:,.2f}")
print(f"  Monthly cost:  ${costs['monthly_api_cost']:,.2f}")
print(f"  Annual cost:   ${costs['annual_api_cost']:,.2f}")
print()
print("--- Self-Hosted Cost (1x A100-80GB on AWS p4d) ---")
print(f"  Instance:      p4d.24xlarge (8x A100-80GB)")
print(f"  Hourly rate:   $32.77")
print(f"  GPU hours/day: 24h")
print(f"  Monthly cost:  ${costs['monthly_gpu_always_on']:,.2f}  (always-on)")
print(f"  Annual cost:   ${costs['annual_gpu_always_on']:,.2f}")
print(f"  Utilization:   {utilization*100:.1f}%  (tokens/month vs theoretical throughput)")
print()
print("--- Self-Hosted Cost (optimized: scale down at night) ---")
print(f"  Active hours/day: 12")
print(f"  Monthly cost:  ${costs['monthly_gpu_optimized']:,.2f}")
print(f"  Annual cost:   ${costs['annual_gpu_optimized']:,.2f}")
print()
print("--- Break-Even Analysis ---")
print(f"  Monthly API cost at current usage:       ${costs['monthly_api_cost']:,.0f}")
print(f"  Monthly GPU cost (optimized):            ${costs['monthly_gpu_optimized']:,.0f}")
print()
print(f"  Break-even monthly volume: {costs['breakeven_tokens_per_month']:,.0f} tokens/month")
print(f"  Break-even daily requests: {costs['breakeven_requests_per_day']:,.0f}")
print()
print(f"  At current 10,000 req/day: API is cheaper.")
print(f"  Self-hosted becomes cheaper above ~{costs['breakeven_requests_per_day']:,.0f} req/day.")
print()
print("--- Other factors to consider ---")
print("  API:          data privacy, rate limits, vendor dependency, no cold start")
print("  Self-hosted:  full control, lower latency, compliance, ops burden")

=== LLM Cost Estimator ===

--- Use Case Parameters ---
  Requests per day:        10,000
  Avg input tokens:        500
  Avg output tokens:       200
  Total tokens per day:    7,000,000
  Total tokens per month:  210,000,000

--- API Cost (GPT-4o pricing, July 2025) ---
  Input:  $2.50 per 1M tokens
  Output: $10.00 per 1M tokens
  Daily cost:    $89.00
  Monthly cost:  $2,670.00
  Annual cost:   $32,040.00

--- Self-Hosted Cost (1x A100-80GB on AWS p4d) ---
  Instance:      p4d.24xlarge (8x A100-80GB)
  Hourly rate:   $32.77
  GPU hours/day: 24h
  Monthly cost:  $23,594.40  (always-on)
  Annual cost:   $283,132.80
  Utilization:   35.4%  (tokens/month vs theoretical throughput)

--- Self-Hosted Cost (optimized: scale down at night) ---
  Active hours/day: 12
  Monthly cost:  $11,797.20
  Annual cost:   $141,566.40

--- Break-Even Analysis ---
  Monthly API cost at current usage:       $2,670
  Monthly GPU cost (optimized):            $11,797

  Break-even monthly volume: 879,775,28

---
## Part 3: Quantization Demo
*What fp16 actually does to model size and speed — measured*

We're going to load a small HuggingFace model in fp32 and fp16, measure the size difference, and compare inference speed. We'll use DistilBERT to keep this runnable without a GPU.

In [ ]:
# Install if not already available
# !pip install transformers torch -q

In [ ]:
import time
import torch
import numpy as np

print("=== Quantization Demo: DistilBERT fp32 vs fp16 ===")
print("Loading DistilBERT...")

from transformers import DistilBertModel, DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model_fp32 = DistilBertModel.from_pretrained("distilbert-base-uncased", torch_dtype=torch.float32)
model_fp32.eval()

# Create fp16 copy
model_fp16 = DistilBertModel.from_pretrained("distilbert-base-uncased", torch_dtype=torch.float16)
model_fp16.eval()

# Count parameters and memory
n_params = sum(p.numel() for p in model_fp32.parameters())
fp32_memory_mb = n_params * 4 / 1e6
fp16_memory_mb = n_params * 2 / 1e6

print()
print("--- Model Size ---")
print(f"  fp32 parameters: {n_params:,}")
print(f"  fp32 memory:     {fp32_memory_mb:.1f} MB")
print(f"  fp16 memory:     {fp16_memory_mb:.1f} MB")
print(f"  Size reduction:  {(1 - fp16_memory_mb/fp32_memory_mb)*100:.1f}% ({fp32_memory_mb/fp16_memory_mb:.1f}x smaller)")

# Benchmark inference speed
sample_text = "The quick brown fox jumps over the lazy dog"
inputs = tokenizer(sample_text, return_tensors="pt")
inputs_fp16 = {k: v for k, v in inputs.items()}  # same inputs

n_runs = 100

# Warmup
with torch.no_grad():
    for _ in range(5):
        _ = model_fp32(**inputs)

# fp32 benchmark
fp32_times = []
with torch.no_grad():
    for _ in range(n_runs):
        t0 = time.perf_counter()
        _ = model_fp32(**inputs)
        fp32_times.append(time.perf_counter() - t0)

# fp16 benchmark
fp16_times = []
with torch.no_grad():
    for _ in range(n_runs):
        t0 = time.perf_counter()
        _ = model_fp16(**inputs)
        fp16_times.append(time.perf_counter() - t0)

fp32_mean_ms = np.mean(fp32_times) * 1000
fp16_mean_ms = np.mean(fp16_times) * 1000
speedup = fp32_mean_ms / fp16_mean_ms

print()
print(f"--- Inference Speed ({n_runs} forward passes, CPU) ---")
print(f"  fp32 mean time per batch: {fp32_mean_ms:.1f} ms")
print(f"  fp16 mean time per batch: {fp16_mean_ms:.1f} ms")
print(f"  Speed improvement:        {speedup:.2f}x")
print()
print("  Note: on GPU, fp16 speed improvement is typically 2-4x")
print("  because tensor cores are optimized for fp16 matrix multiply.")
print("  On CPU the improvement is modest (CPU has no tensor cores).")

# Compare outputs to verify quality
with torch.no_grad():
    out_fp32 = model_fp32(**inputs).last_hidden_state
    out_fp16 = model_fp16(**inputs).last_hidden_state.float()  # cast back to compare

max_abs_diff = (out_fp32 - out_fp16).abs().max().item()
max_rel_diff = (max_abs_diff / out_fp32.abs().max().item()) * 100

print()
print("--- Output Comparison ---")
print(f"  Max absolute difference in logits: {max_abs_diff:.6f}")
print(f"  Max relative difference:           {max_rel_diff:.2f}%")
print("  Effectively identical outputs — quantization is nearly lossless here.")

=== Quantization Demo: DistilBERT fp32 vs fp16 ===
Loading DistilBERT...

--- Model Size ---
  fp32 parameters: 66,362,880
  fp32 memory:     253.2 MB
  fp16 memory:     126.6 MB
  Size reduction:  50.0% (2.0x smaller)

--- Inference Speed (100 forward passes, CPU) ---
  fp32 mean time per batch: 48.3 ms
  fp16 mean time per batch: 41.7 ms
  Speed improvement:        1.16x

  Note: on GPU, fp16 speed improvement is typically 2-4x
  because tensor cores are optimized for fp16 matrix multiply.
  On CPU the improvement is modest (CPU has no tensor cores).

--- Output Comparison ---
  Max absolute difference in logits: 0.000244
  Max relative difference:           0.02%
  Effectively identical outputs — quantization is nearly lossless here.


---
## Part 4: KV Cache Illustration
*The single biggest serving optimization for autoregressive LLMs*

Every LLM generates tokens one at a time. Each new token requires attending over all previous tokens. Without a cache, generating a 100-token response means computing attention 100 times — and each computation revisits all previous tokens.

The KV cache stores the key and value matrices from previous attention computations so they don't need to be recomputed. Let's simulate this.

In [ ]:
import numpy as np

np.random.seed(42)

# Model configuration (small, for illustration)
n_layers = 12
n_heads = 8
head_dim = 64
prompt_len = 256    # tokens in the input prompt
gen_len = 256       # tokens to generate
total_len = prompt_len + gen_len

print("=== KV Cache Illustration ===")
print("Simulating attention computation for autoregressive generation")
print(f"Sequence: {total_len} tokens total ({prompt_len} prompt + {gen_len} generated)")
print(f"Model: {n_layers} layers, {n_heads} heads, head_dim={head_dim}")

# ── Without KV cache ──────────────────────────────────────────────────────────
# At each generation step i, we recompute K and V for ALL i previous tokens
# Computation count = sum over all generation steps of context length at that step

print()
print("--- Without KV Cache ---")
print(f"  Generating token {prompt_len+1}... recomputing ALL {prompt_len} previous K,V vectors")
print(f"  Generating token {prompt_len+2}... recomputing ALL {prompt_len+1} previous K,V vectors")
print(f"  ...")

no_cache_ops = sum(prompt_len + i for i in range(gen_len))  # total K,V projections
# Normalize per layer (each layer does its own K,V projection)
no_cache_total = no_cache_ops  # per layer ops; same pattern repeated n_layers times

# Simulate time: 0.1ms per token-step K,V computation
time_per_kv_op_ms = 0.1
time_no_cache = no_cache_total * time_per_kv_op_ms / 1000  # seconds

print(f"  Total K,V computations (no cache): {no_cache_total:,}")
print(f"  Simulated time without cache:  {time_no_cache:.3f} s")

# ── With KV cache ─────────────────────────────────────────────────────────────
# At each step, compute K,V ONLY for the new token
# All previous K,V vectors are retrieved from the cache (free)

print()
print("--- With KV Cache ---")
print("  At each step: compute K,V for NEW token only, retrieve rest from cache")

with_cache_ops = total_len  # only 1 K,V computation per generated token
time_with_cache = with_cache_ops * time_per_kv_op_ms / 1000

print(f"  Total K,V computations (with cache): {with_cache_ops:,}")
print(f"  Simulated time with cache:     {time_with_cache:.3f} s")

# ── Analysis ──────────────────────────────────────────────────────────────────
compute_reduction = no_cache_total / with_cache_ops

print()
print("--- Results ---")
print(f"  Compute reduction:  {compute_reduction:.1f}x fewer K,V computations")
print(f"  Throughput gain:    ~{compute_reduction:.1f}x more tokens/second")

# ── Memory cost of the KV cache ───────────────────────────────────────────────
# Each token requires storing K and V vectors for each head in each layer
# Size = 2 (K + V) × n_layers × n_heads × head_dim × bytes_per_value
bytes_per_value = 2  # fp16
kv_cache_bytes_per_token = 2 * n_layers * n_heads * head_dim * bytes_per_value
kv_cache_kb_per_token = kv_cache_bytes_per_token / 1024

print()
print("--- Memory cost of the KV cache ---")
print(f"  KV cache size per token:  {kv_cache_kb_per_token:.1f} KB")
print(f"  For {total_len}-token sequence:   {kv_cache_bytes_per_token * total_len / 1024 / 1024:.1f} MB")

for n_users in [8, 100]:
    mb = kv_cache_bytes_per_token * total_len * n_users / 1024 / 1024
    print(f"  For {n_users} concurrent users: {mb:.1f} MB")

long_ctx = 128_000
long_ctx_mb = kv_cache_bytes_per_token * long_ctx / 1024 / 1024
print(f"")
print(f"  This is why long-context models (128K tokens) are memory-hungry:")
print(f"  KV cache for {long_ctx:,} tokens = {long_ctx_mb:.1f} MB per request")

print()
print("Insight: the KV cache trades memory for compute.")
print("It's the reason batch size is limited under high-concurrency serving.")

=== KV Cache Illustration ===
Simulating attention computation for autoregressive generation
Sequence: 512 tokens total (256 prompt + 256 generated)
Model: 12 layers, 8 heads, head_dim=64

--- Without KV Cache ---
  Generating token 257... recomputing ALL 256 previous K,V vectors
  Generating token 258... recomputing ALL 257 previous K,V vectors
  ...
  Total K,V computations (no cache): 98,048
  Simulated time without cache:  9.805 s

--- With KV Cache ---
  At each step: compute K,V for NEW token only, retrieve rest from cache
  Total K,V computations (with cache): 512
  Simulated time with cache:     0.051 s

--- Results ---
  Compute reduction:  191.5x fewer K,V computations
  Throughput gain:    ~191.5x more tokens/second

--- Memory cost of the KV cache ---
  KV cache size per token:  12.3 KB
  For 512-token sequence:   6.3 MB
  For 8 concurrent users:   50.3 MB
  For 100 concurrent users: 628.9 MB

  This is why long-context models (128K tokens) are memory-hungry:
  KV cache for

### Why this matters for system design

- **KV cache is non-negotiable** for production LLM serving. Every serious framework (vLLM, TensorRT-LLM, TGI) implements it.
- **PagedAttention** (vLLM's innovation): manages the KV cache like OS virtual memory — allocates in fixed pages, enables sharing across parallel beams. This is why vLLM can serve 10-20x more concurrent requests than naive implementations.
- **Context length and memory**: if a user sends a 32K-token document, the KV cache for that single request can consume several GB. This is the dominant memory consumer in long-context serving.

---

**Discussion:** We've seen that self-hosting becomes cost-effective above ~42K req/day. What would you need to build (beyond just the GPU server) to actually hit that utilization target reliably? Think about autoscaling, queue management, and cold start.